In [1]:
import editor

import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# settings
array_size = 128

nn_kernel_size = 9
nn_hidden_dim = 10

In [3]:
class Net(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.fc1 = nn.Linear(kernel_size * kernel_size, nn_hidden_dim)
        self.fc2 = nn.Linear(nn_hidden_dim, nn_hidden_dim)
        self.fc3 = nn.Linear(nn_hidden_dim, nn_hidden_dim)
        self.fc4 = nn.Linear(nn_hidden_dim, nn_hidden_dim)
        self.fc5 = nn.Linear(nn_hidden_dim, nn_hidden_dim)
        self.fc6 = nn.Linear(nn_hidden_dim, 1)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.tanh(self.fc1(x))
        x = F.tanh(self.fc2(x))
        x = F.tanh(self.fc3(x))
        x = F.tanh(self.fc4(x))
        x = F.tanh(self.fc5(x))
        x = F.tanh(self.fc6(x))
        x = torch.abs(x)
        x -= x.min()
        x /= x.max()
        
        return x

In [4]:
class TempSim:
    def __init__(self):
        self.model = Net(nn_kernel_size)
    
    def extract_kernel(self, global_tensor: torch.tensor, start_x, end_x, start_y, end_y):
        return global_tensor[:, start_x:end_x, start_y:end_y]
    
    def step(self, state: torch.tensor):
        _, H, W = state.shape
        p = nn_kernel_size // 2
        x = F.pad(state[None], (p, p, p, p), mode="circular")  # wrap edges
        patches = F.unfold(x, nn_kernel_size).transpose(1, 2)[0]  # (H*W, k*k)
        return self.model(patches).reshape(1, H, W)

In [5]:
sim = TempSim()

In [6]:
editor = editor.NCAEditor(sim, torch.rand((array_size, array_size)))

In [7]:
editor.run()

In [8]:
def extract_kernel(global_tensor: torch.tensor, start_x, end_x, start_y, end_y):
    return global_tensor[:, start_x:end_x, start_y:end_y]

In [9]:
startx = 0
starty = 0

endx = 2
endy = 2

In [10]:
x = torch.tensor(((1, 2, 3, 4),
                  (5, 6, 7, 8),
                  (9, 0, 1, 2),
                  (3, 4, 5, 6))).unsqueeze(0)

In [11]:
extract_kernel(x, 1, 4, 1, 4)

tensor([[[6, 7, 8],
         [0, 1, 2],
         [4, 5, 6]]])